<a href="https://colab.research.google.com/github/nikamsudarshan/movie-recommendation-engine/blob/main/movie_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re
import pandas as pd
import numpy as np

def clean_text(text):
    """Applies lowercasing, strips punctuation, and normalizes whitespaces."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_metadata_soup(df):
    """Combines core descriptive text attributes into a single feature column."""
    df = df.copy()
    # Using 'genres' and 'original_title' as standard proxy fields for standard datasets
    df['genres'] = df.get('genres', pd.Series([''] * len(df))).fillna('')
    df['overview'] = df.get('overview', pd.Series([''] * len(df))).fillna('')

    df['metadata_soup'] = df['overview'] + " " + df['genres']
    df['cleaned_soup'] = df['metadata_soup'].apply(clean_text)
    return df

def generate_interaction_matrix(ratings_df):
    """Converts a long ratings dataframe into a sparse pivot table for KNN."""
    interaction_matrix = ratings_df.pivot(
        index='userId',
        columns='movieId',
        values='rating'
    ).fillna(0)
    return interaction_matrix

In [6]:
import pandas as pd
import urllib.request
import zipfile

# 1. Fetch the official MovieLens dataset (Small version) directly from the source
print("Downloading official dataset from GroupLens...")
zip_url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
urllib.request.urlretrieve(zip_url, "ml-latest-small.zip")

# Extract the zip file in the Colab environment
with zipfile.ZipFile("ml-latest-small.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Load the newly extracted CSVs
movies_df = pd.read_csv("ml-latest-small/movies.csv")
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")

# 2. Process the Data
print("Processing Text Pipeline...")
# MovieLens groups genres with the '|' character (e.g., Action|Adventure|Sci-Fi).
# We will clean that up to build our text soup.
movies_df['overview'] = movies_df['genres'].str.replace('|', ' ')
processed_movies = build_metadata_soup(movies_df)

print("Processing Interaction Matrix...")
interaction_grid = generate_interaction_matrix(ratings_df)

# 3. View the Results
print("\n--- Processed Text Data Sample ---")
print(processed_movies[['title', 'cleaned_soup']].head(3))

print("\n--- Interaction Matrix Structural Shape ---")
print(f"Users: {interaction_grid.shape[0]} | Movies: {interaction_grid.shape[1]}")

Processing Text Pipeline...
Processing Interaction Matrix...

--- Processed Text Data Sample ---
                     title                                       cleaned_soup
0         Toy Story (1995)  adventure animation children comedy fantasy ad...
1           Jumanji (1995)  adventure children fantasy adventurechildrenfa...
2  Grumpier Old Men (1995)                       comedy romance comedyromance

--- Interaction Matrix Structural Shape ---
Users: 610 | Movies: 9724
